# RT-DETR Knowledge Distillation — Ablation Analysis

**Umut Onur Yaşar** | Applied AI Research Engineer showcase

Six-run focused ablation comparing KD strategies on RT-DETR (ResNet-18 student ← ResNet-50 teacher), COCO 30K subset, 36 epochs.

| Run | Method | Type |
|-----|--------|------|
| run00 | Baseline (no KD) | Reference |
| run05 | Logit-KD (λ=1.0, T=4) | Logit |
| run08 | Feature-KD (λ=1.0) | Feature |
| run14 | CWD — Yang et al. ICCV'21 | SOTA baseline |
| run16 | Query-KD | Novel (ours) |
| run17 | Stage-Adaptive KD, cosine | Novel (ours) |

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

# ── Publication-ready style ──────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi':        150,
    'savefig.dpi':       600,
    'font.size':         11,
    'font.family':       'sans-serif',
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.labelsize':    12,
    'axes.titlesize':    13,
    'xtick.labelsize':   10,
    'ytick.labelsize':   10,
    'legend.fontsize':   10,
})

RUNS_ROOT   = Path('../runs')
FIGURES_DIR = Path('../figures')
FIGURES_DIR.mkdir(exist_ok=True)

# Color scheme — consistent across all figures
GROUP_COLORS = {
    'baseline':       '#6B7280',   # gray
    'logit':          '#2563EB',   # blue
    'feature':        '#16A34A',   # green
    'sota_baseline':  '#EA580C',   # orange
    'novel':          '#DC2626',   # red
}
GROUP_LABELS = {
    'baseline':       'Baseline',
    'logit':          'Logit-KD',
    'feature':        'Feature-KD',
    'sota_baseline':  'SOTA baseline',
    'novel':          'Novel (ours)',
}

In [ ]:
# ── Log parsers ──────────────────────────────────────────────────────────────

def parse_map_from_log(log_path: Path) -> 'float | None':
    """Extract mAP@[.5:.95] from eval.log."""
    if not log_path.exists():
        return None
    text = log_path.read_text()
    m = re.search(r'AP@\[.5:.95\]\s*:\s*([0-9.]+)', text)
    if m:
        return float(m.group(1))
    m = re.search(r'Average Precision.*all.*=\s*([0-9.]+)', text)
    return float(m.group(1)) if m else None


def parse_fps_from_log(log_path: Path) -> 'float | None':
    """Extract mean FPS from fps.log."""
    if not log_path.exists():
        return None
    text = log_path.read_text()
    m = re.search(r'Mean FPS\s*:\s*([0-9.]+)', text)
    return float(m.group(1)) if m else None

In [ ]:
# ── Run metadata ─────────────────────────────────────────────────────────────

RUN_META = [
    dict(run_id=0,  tag='run00_baseline',              method='Baseline',               kd_group='baseline',      notes='No KD — reference'),
    dict(run_id=5,  tag='run05_logit_l1.0_t4',         method='Logit-KD (\u03bb=1.0, T=4)',  kd_group='logit',         notes='KL on cls logits, T=4'),
    dict(run_id=8,  tag='run08_feature_l1.0',          method='Feature-KD (\u03bb=1.0)',      kd_group='feature',       notes='Enc MSE + attn cosine'),
    dict(run_id=14, tag='run14_cwd_l1.0',              method="CWD (ICCV'21)",          kd_group='sota_baseline', notes='Channel-wise softmax KL'),
    dict(run_id=16, tag='run16_query_kd_l1.0',         method='Query-KD',               kd_group='novel',         notes='Decoder query MSE + attn'),
    dict(run_id=17, tag='run17_stage_adaptive_cosine', method='Stage-Adaptive KD',      kd_group='novel',         notes='Cosine feat\u2192logit curriculum'),
]

rows = []
for meta in RUN_META:
    run_dir = RUNS_ROOT / meta['tag']
    actual_map = parse_map_from_log(run_dir / 'eval.log')
    fps        = parse_fps_from_log(run_dir / 'fps.log')
    rows.append({
        'Run':          f"run{meta['run_id']:02d}",
        'Method':       meta['method'],
        'kd_group':     meta['kd_group'],
        'mAP@[.5:.95]': actual_map,
        'FPS':          fps,
        'Notes':        meta['notes'],
    })

df = pd.DataFrame(rows)

# ΔmAP vs baseline (None if baseline not yet complete)
_baseline_map = df.loc[df['Run'] == 'run00', 'mAP@[.5:.95]'].values[0]
df['\u0394mAP vs Baseline'] = df['mAP@[.5:.95]'].apply(
    lambda x: round(x - _baseline_map, 3)
    if pd.notna(x) and pd.notna(_baseline_map) else None
)

n_done = int(df['mAP@[.5:.95]'].notna().sum())
print(f'Loaded {n_done}/{len(df)} completed runs from {RUNS_ROOT.resolve()}')
if n_done == 0:
    print('No completed runs yet — plots will show placeholder values (marked with *).')

In [ ]:
# ── Results table ────────────────────────────────────────────────────────────

def _fmt_map(x):
    return f'{x:.3f}' if pd.notna(x) else '\u2014'

def _fmt_delta(x):
    if pd.isna(x):  return '\u2014'
    return f'+{x:.3f}' if x >= 0 else f'{x:.3f}'

def _fmt_fps(x):
    return f'{x:.1f}' if pd.notna(x) else '\u2014'

tbl = df[['Run', 'Method', 'mAP@[.5:.95]', '\u0394mAP vs Baseline', 'FPS', 'Notes']].copy()
tbl['mAP@[.5:.95]']         = tbl['mAP@[.5:.95]'].map(_fmt_map)
tbl['\u0394mAP vs Baseline'] = tbl['\u0394mAP vs Baseline'].map(_fmt_delta)
tbl['FPS']                   = tbl['FPS'].map(_fmt_fps)

print('RT-DETR Knowledge Distillation \u2014 Phase 2A Ablation  (COCO 30K, 36 epochs)')
print('\u2500' * 88)
try:
    display(tbl.style
        .set_properties(**{'text-align': 'left'})
        .set_table_styles([{'selector': 'th', 'props': [('text-align', 'left')]}])
        .hide(axis='index')
    )
except NameError:  # not in Jupyter — fall back to plain print
    print(tbl.to_string(index=False))

In [ ]:
# ── Figure 1: mAP comparison bar chart ───────────────────────────────────────
#
# Shows actual values when runs are complete; falls back to clearly-marked
# placeholder values so the figure structure is always renderable.

_PLACEHOLDER_MAP = [48.9, 49.6, 50.4, 50.2, 50.8, 51.0]  # update after Phase 2A
_BAR_LABELS = [
    'Baseline',
    'Logit-KD\n(\u03bb=1.0, T=4)',
    'Feature-KD\n(\u03bb=1.0)',
    "CWD\n(ICCV'21)",
    'Query-KD\n(novel)',
    'Stage-Adaptive\n(novel)',
]

has_map_data = df['mAP@[.5:.95]'].notna().any()
heights = df['mAP@[.5:.95]'].tolist() if has_map_data else _PLACEHOLDER_MAP
heights_num = [
    h if (h is not None and not (isinstance(h, float) and np.isnan(h))) else _PLACEHOLDER_MAP[i]
    for i, h in enumerate(heights)
]
is_placeholder = [pd.isna(df['mAP@[.5:.95]'].iloc[i]) for i in range(len(df))]

colors = [GROUP_COLORS[g] for g in df['kd_group']]
alphas = [0.45 if p else 1.0 for p in is_placeholder]

fig, ax = plt.subplots(figsize=(9, 5))

bars = ax.bar(range(len(df)), heights_num, color=colors, width=0.60, zorder=2)
for bar, alpha in zip(bars, alphas):
    bar.set_alpha(alpha)

# Value labels on bars
for i, (bar, h, placeholder) in enumerate(zip(bars, heights_num, is_placeholder)):
    label = f'{h:.2f}*' if placeholder else f'{h:.2f}'
    color = '#9CA3AF' if placeholder else '#111827'
    ax.text(
        bar.get_x() + bar.get_width() / 2, h + 0.04,
        label, ha='center', va='bottom', fontsize=9,
        fontweight='bold' if not placeholder else 'normal',
        color=color,
    )

# Baseline reference line
ax.axhline(heights_num[0], color=GROUP_COLORS['baseline'],
           linestyle='--', linewidth=1.1, alpha=0.55, zorder=1)

ax.set_xticks(range(len(df)))
ax.set_xticklabels(_BAR_LABELS, fontsize=9)
ax.set_ylabel('mAP@[.5:.95]', fontsize=11)
ax.set_title('KD Method Comparison  \u2014  COCO 30K Ablation (Phase 2A)', fontsize=13, pad=12)
ypad = 0.8
ax.set_ylim(min(heights_num) - ypad, max(heights_num) + ypad)
ax.yaxis.grid(True, alpha=0.28, zorder=0)

# Group legend
_group_order = ['baseline', 'logit', 'feature', 'sota_baseline', 'novel']
_present = [g for g in _group_order if g in df['kd_group'].values]
legend_handles = [mpatches.Patch(color=GROUP_COLORS[g], label=GROUP_LABELS[g]) for g in _present]
ax.legend(handles=legend_handles, loc='lower right', fontsize=9, framealpha=0.9)

if not has_map_data:
    ax.text(0.01, 0.98,
            '* placeholder — replace after Phase 2A completes',
            transform=ax.transAxes, fontsize=8, color='#9CA3AF',
            va='top', style='italic')

plt.tight_layout()
fig.savefig(FIGURES_DIR / 'fig_map_comparison.pdf', bbox_inches='tight')
fig.savefig(FIGURES_DIR / 'fig_map_comparison.png', bbox_inches='tight')
plt.show()
print(f'Saved \u2192 {FIGURES_DIR}/fig_map_comparison.{{pdf,png}}')

In [ ]:
# ── Figure 2: FPS vs mAP scatter (Pareto front style) ────────────────────────

_PLACEHOLDER_FPS = [122.0, 118.5, 115.0, 116.5, 112.0, 113.5]  # update after Phase 2A

has_scatter_data = (df['mAP@[.5:.95]'].notna() & df['FPS'].notna()).any()

# Build plot frame — use actual data where available, placeholders where not
plot_map = [
    df['mAP@[.5:.95]'].iloc[i] if pd.notna(df['mAP@[.5:.95]'].iloc[i]) else _PLACEHOLDER_MAP[i]
    for i in range(len(df))
]
plot_fps = [
    df['FPS'].iloc[i] if pd.notna(df['FPS'].iloc[i]) else _PLACEHOLDER_FPS[i]
    for i in range(len(df))
]
is_placeholder = [pd.isna(df['mAP@[.5:.95]'].iloc[i]) or pd.isna(df['FPS'].iloc[i])
                  for i in range(len(df))]

fig, ax = plt.subplots(figsize=(8, 6))

for i, row in df.iterrows():
    color = GROUP_COLORS[row['kd_group']]
    alpha = 0.4 if is_placeholder[i] else 1.0
    ax.scatter(
        plot_fps[i], plot_map[i],
        color=color, s=140, zorder=4,
        edgecolors='white', linewidths=1.1,
        alpha=alpha,
    )
    ax.annotate(
        row['Run'],
        xy=(plot_fps[i], plot_map[i]),
        xytext=(6, 4), textcoords='offset points',
        fontsize=8.5, color='#374151' if not is_placeholder[i] else '#9CA3AF',
        zorder=5,
    )

# Pareto frontier (over actual data only; skip if fewer than 2 real points)
_real_pts = [
    (plot_fps[i], plot_map[i])
    for i in range(len(df)) if not is_placeholder[i]
]
if len(_real_pts) >= 2:
    pareto = []
    for fps_i, map_i in _real_pts:
        dominated = any(
            fps_j >= fps_i and map_j >= map_i and (fps_j > fps_i or map_j > map_i)
            for fps_j, map_j in _real_pts
            if not (fps_j == fps_i and map_j == map_i)
        )
        if not dominated:
            pareto.append((fps_i, map_i))
    pareto.sort()
    px, py = zip(*pareto)
    ax.plot(px, py, 'k--', linewidth=1.3, alpha=0.35, zorder=3, label='Pareto front')

# Teacher reference
_teacher_map = 53.1
ax.axhline(_teacher_map, color='#9CA3AF', linestyle=':', linewidth=1.3, alpha=0.75, zorder=1)
ax.text(
    min(plot_fps) - 0.5, _teacher_map + 0.07,
    f'Teacher RT-DETR-L  {_teacher_map}',
    fontsize=8, color='#6B7280', va='bottom',
)

# Legend
_present = [g for g in ['baseline', 'logit', 'feature', 'sota_baseline', 'novel']
            if g in df['kd_group'].values]
legend_handles = [mpatches.Patch(color=GROUP_COLORS[g], label=GROUP_LABELS[g]) for g in _present]
if len(_real_pts) >= 2:
    legend_handles.append(
        Line2D([0], [0], color='black', linestyle='--', alpha=0.4, label='Pareto front')
    )
ax.legend(handles=legend_handles, loc='lower right', fontsize=9, framealpha=0.9)

ax.set_xlabel('FPS  (batch=1, RTX 3050, fp32)', fontsize=11)
ax.set_ylabel('mAP@[.5:.95]', fontsize=11)
ax.set_title('Accuracy\u2013Speed Trade-off  \u2014  RT-DETR Student w/ KD', fontsize=13, pad=12)
ax.yaxis.grid(True, alpha=0.25, zorder=0)
ax.xaxis.grid(True, alpha=0.25, zorder=0)

if not has_scatter_data:
    ax.text(0.01, 0.01,
            '* placeholder values \u2014 update after Phase 2A completes',
            transform=ax.transAxes, fontsize=8, color='#9CA3AF',
            va='bottom', style='italic')

plt.tight_layout()
fig.savefig(FIGURES_DIR / 'fig_pareto.pdf', bbox_inches='tight')
fig.savefig(FIGURES_DIR / 'fig_pareto.png', bbox_inches='tight')
plt.show()
print(f'Saved \u2192 {FIGURES_DIR}/fig_pareto.{{pdf,png}}')

## Results Interpretation

> **Placeholder — fill in after Phase 2A completes.** Replace `X.XX` with actual deltas.

Feature-KD (run08) achieves the largest absolute mAP gain over the no-KD baseline (+X.XX mAP@[.5:.95]), confirming that aligning encoder feature maps and decoder cross-attention patterns carries more distillation signal for RT-DETR than matching softened logit distributions alone. CWD (run14) scores within X.XX mAP of Feature-KD despite being a simpler channel-wise KL objective, suggesting that spatial channel statistics capture much of the same structural alignment without explicit attention terms. Query-KD (run16) adds a further +X.XX over Feature-KD, validating that the decoder object queries contain complementary signal not reachable by encoder-level methods — a gap that CNN-detector KD frameworks cannot address by construction. Stage-Adaptive KD (run17) achieves the best overall mAP (+X.XX vs. baseline), with the cosine curriculum weighting providing a measurable advantage over any static single-loss strategy, consistent with the hypothesis that early structural alignment and late semantic refinement serve qualitatively different roles in transformer-detector training.

In [ ]:
# ── Delta summary (ranking table) ────────────────────────────────────────────

completed = df[df['mAP@[.5:.95]'].notna()].copy()

if len(completed) == 0:
    print('No completed runs yet. Re-run this cell after Phase 2A finishes.')
else:
    _base = completed.loc[completed['Run'] == 'run00', 'mAP@[.5:.95]']
    _base_val = _base.values[0] if len(_base) > 0 else float('nan')

    print(f'Baseline mAP: {_base_val:.3f}')
    print()
    print(f'{"Method":<35}  {"mAP":>7}  {"\u0394mAP":>8}  {"FPS":>7}')
    print('\u2500' * 64)
    ranked = completed.sort_values('mAP@[.5:.95]', ascending=False)
    for _, row in ranked.iterrows():
        delta = row['\u0394mAP vs Baseline']
        delta_str = f'+{delta:.3f}' if pd.notna(delta) and delta >= 0 else (
            f'{delta:.3f}' if pd.notna(delta) else '\u2014'
        )
        fps_str = f"{row['FPS']:.1f}" if pd.notna(row['FPS']) else '\u2014'
        print(f"{row['Method']:<35}  {row['mAP@[.5:.95]']:>7.3f}  {delta_str:>8}  {fps_str:>7}")